# Stage 2 - Metric Foundation Tests (S2.4)

Validates the evaluation metrics used by the U-Net vs V-Net decoder comparison **before** any
real training, using synthetic masks with analytically known answers. Covers:

- Per-bone **Dice / IoU** and **HD95 / ASSD** (mm), plus macro-per-bone and subject-level aggregation.
- **Empty-prediction** and **finite-failure** handling (surface metrics -> NaN, not a crash).
- **Known-offset** exactness and mm-conversion / spacing-linearity.
- **Anisotropic-spacing guard**: the scalar-spacing surface metric is only valid on isotropic
  (256^3 -> 0.78125 mm) grids; raw anisotropic data must be resampled first.
- **Connected-component agreement**, **min-gap error** and **false bridging** for fractures.

The metric functions below are **copied verbatim** from `03_decoder_pipeline.ipynb` (the project is
notebook-first, so they are not importable). If they change there, update the copies here to match.

Run: `.venv/Scripts/python.exe -m jupyter nbconvert --to notebook --execute --inplace notebooks/modeling/tests/metric_foundation.ipynb`
(or run all cells). The final cell raises `AssertionError` if any check fails.

In [1]:
import math
import numpy as np
import torch
import pandas as pd
from scipy.ndimage import binary_erosion, distance_transform_edt, label

BONES = ["femur", "tibia", "patella", "fibula"]
N_CLASSES = 4

# ---- verbatim from 03_decoder_pipeline.ipynb (keep in sync) ----
def _surface_dists(pred_bin, gt_bin, spacing):
    sp = pred_bin & ~binary_erosion(pred_bin); sg = gt_bin & ~binary_erosion(gt_bin)
    if sp.sum() == 0 or sg.sum() == 0:
        return None
    dg = distance_transform_edt(~sg) * spacing
    dp = distance_transform_edt(~sp) * spacing
    return dg[sp], dp[sg]

def hd95(pred_bin, gt_bin, spacing=1.0):
    d = _surface_dists(pred_bin, gt_bin, spacing)
    return float("nan") if d is None else float(np.percentile(np.concatenate(d), 95))

def assd(pred_bin, gt_bin, spacing=1.0):
    d = _surface_dists(pred_bin, gt_bin, spacing)
    return float("nan") if d is None else float(np.concatenate(d).mean())

def per_bone_dice_iou(logits, target, thr=0.5):
    p = (torch.sigmoid(logits.float()) > thr).float()
    t = (target > 0.5).float()
    p = p.reshape(p.size(0), p.size(1), -1); t = t.reshape(t.size(0), t.size(1), -1)
    inter = (p * t).sum(-1); psum = p.sum(-1); tsum = t.sum(-1)
    dice = (2 * inter + 1e-6) / (psum + tsum + 1e-6)
    iou = (inter + 1e-6) / (psum + tsum - inter + 1e-6)
    return dice.cpu().numpy(), iou.cpu().numpy()

In [2]:
# ---- test-only helpers (connected components, min-gap, mask->logits, check harness) ----
def count_components(mask):
    return int(label(mask)[1])

def min_gap_voxels(mask_a, mask_b):
    """Min distance (voxels) from mask_a to mask_b."""
    if mask_a.sum() == 0 or mask_b.sum() == 0:
        return float("nan")
    return float(distance_transform_edt(~mask_b)[mask_a].min())

def logits_from_mask(mask, pos=6.0, neg=-6.0):
    return torch.where(torch.as_tensor(mask), torch.tensor(pos), torch.tensor(neg)).float()

def one_voxel(shape, idx):
    m = np.zeros(shape, bool); m[idx] = True; return m

def assd_sampling(pred_bin, gt_bin, sampling):
    """Anisotropy-aware ASSD (uses distance_transform_edt sampling=), for the guard test only."""
    sp = pred_bin & ~binary_erosion(pred_bin); sg = gt_bin & ~binary_erosion(gt_bin)
    if sp.sum() == 0 or sg.sum() == 0:
        return float("nan")
    dg = distance_transform_edt(~sg, sampling=sampling); dp = distance_transform_edt(~sp, sampling=sampling)
    return float(np.concatenate([dg[sp], dp[sg]]).mean())

RESULTS = []
def check(name, cond, detail=""):
    RESULTS.append((name, bool(cond), detail))
    print(("PASS" if cond else "FAIL"), "-", name, ("| " + detail) if detail else "")

## Case 1 - empty prediction & finite-failure handling

In [3]:
gt = np.zeros((32, 32, 32), bool); gt[10:15, 10:15, 10:15] = True
pred_empty = np.zeros_like(gt)
d, i = per_bone_dice_iou(logits_from_mask(pred_empty)[None, None], torch.as_tensor(gt)[None, None].float())
check("empty-pred dice==0", abs(d[0, 0]) < 1e-3, f"dice={d[0,0]:.4g}")
check("empty-pred iou==0", abs(i[0, 0]) < 1e-3, f"iou={i[0,0]:.4g}")
check("empty-pred hd95 is NaN", math.isnan(hd95(pred_empty, gt, 0.78125)))
check("empty-pred assd is NaN", math.isnan(assd(pred_empty, gt, 0.78125)))
check("perfect dice==1", abs(per_bone_dice_iou(logits_from_mask(gt)[None, None], torch.as_tensor(gt)[None, None].float())[0][0, 0] - 1) < 1e-3)

PASS - empty-pred dice==0 | dice=8e-09
PASS - empty-pred iou==0 | iou=8e-09
PASS - empty-pred hd95 is NaN 
PASS - empty-pred assd is NaN 
PASS - perfect dice==1 


## Case 2 - known offset (exact) & mm-conversion / spacing linearity
Single-voxel masks give an exact symmetric surface distance: `assd == hd95 == dx * spacing`.

In [4]:
shape = (40, 40, 40)
gt_v = one_voxel(shape, (20, 20, 20))
for dx in (1, 3, 7):
    pr = one_voxel(shape, (20 + dx, 20, 20))
    check(f"offset dx={dx} assd==dx (voxels)", abs(assd(pr, gt_v, 1.0) - dx) < 1e-9, f"assd={assd(pr,gt_v,1.0)}")
    check(f"offset dx={dx} hd95==dx (voxels)", abs(hd95(pr, gt_v, 1.0) - dx) < 1e-9, f"hd95={hd95(pr,gt_v,1.0)}")
pr3 = one_voxel(shape, (23, 20, 20))
check("mm-scaling assd(0.78125)==3*0.78125", abs(assd(pr3, gt_v, 0.78125) - 3 * 0.78125) < 1e-9, f"assd_mm={assd(pr3,gt_v,0.78125):.5f}")
check("mm-scaling linear in spacing", abs(assd(pr3, gt_v, 2.0) - 2 * assd(pr3, gt_v, 1.0)) < 1e-9)
check("assd monotonic in offset", assd(one_voxel(shape,(21,20,20)),gt_v,1.0) < assd(one_voxel(shape,(27,20,20)),gt_v,1.0))

PASS - offset dx=1 assd==dx (voxels) | assd=1.0
PASS - offset dx=1 hd95==dx (voxels) | hd95=1.0
PASS - offset dx=3 assd==dx (voxels) | assd=3.0
PASS - offset dx=3 hd95==dx (voxels) | hd95=3.0
PASS - offset dx=7 assd==dx (voxels) | assd=7.0
PASS - offset dx=7 hd95==dx (voxels) | hd95=7.0
PASS - mm-scaling assd(0.78125)==3*0.78125 | assd_mm=2.34375
PASS - mm-scaling linear in spacing 
PASS - assd monotonic in offset 


## Case 3 - anisotropic-spacing guard
The reused surface metric takes a **single scalar** spacing, so it is only correct on isotropic
grids. With a diagonal (through-plane + in-plane) displacement, no single scalar reproduces the
true anisotropic distance -> raw anisotropic data must be resampled to 256^3 isotropic first.

In [5]:
aniso = (3.0, 0.7, 0.7)  # z=3.0mm, y=x=0.7mm (Ruikar thick-slice, pre-resample)
gt_a = one_voxel(shape, (20, 20, 20)); pr_az = one_voxel(shape, (23, 20, 23))  # diagonal: dz=3, dx=3
scalar_val = assd(pr_az, gt_a, 0.7)              # naive: in-plane spacing as one scalar (wrong for z)
aware_val = assd_sampling(pr_az, gt_a, aniso)     # correct anisotropic mm distance
check("anisotropy needs resample (scalar != sampling-aware)", abs(scalar_val - aware_val) > 1e-6, f"scalar={scalar_val:.4f} aware={aware_val:.4f}")
iso = (0.78125, 0.78125, 0.78125)
check("isotropic: scalar == sampling-aware", abs(assd(pr_az, gt_a, 0.78125) - assd_sampling(pr_az, gt_a, iso)) < 1e-9)

PASS - anisotropy needs resample (scalar != sampling-aware) | scalar=2.9698 aware=9.2418
PASS - isotropic: scalar == sampling-aware 


## Case 4 - bridged fracture: connected-component agreement, min-gap, false bridging

In [6]:
frac = np.zeros((40, 40, 40), bool)
frac[15:20, 18:22, 18:22] = True          # fragment A
frac[24:29, 18:22, 18:22] = True          # fragment B (gap at x=20..23)
check("fractured GT has 2 components", count_components(frac) == 2)
pred_faithful = frac.copy()
check("faithful pred has 2 components (CC agreement)", count_components(pred_faithful) == count_components(frac))
pred_bridge = frac.copy(); pred_bridge[20:24, 18:22, 18:22] = True   # fill the gap
check("bridged pred collapses to 1 component (false bridging detected)", count_components(pred_bridge) == 1)
A = np.zeros_like(frac); A[15:20, 18:22, 18:22] = True
B = np.zeros_like(frac); B[24:29, 18:22, 18:22] = True
gap = min_gap_voxels(A, B)
check("GT fragment gap > 0 (min-gap error preserved)", gap > 0, f"gap={gap} voxels")
check("bridging removes the gap (CC agreement flags it)", count_components(pred_bridge) != count_components(frac))

PASS - fractured GT has 2 components 
PASS - faithful pred has 2 components (CC agreement) 
PASS - bridged pred collapses to 1 component (false bridging detected) 
PASS - GT fragment gap > 0 (min-gap error preserved) | gap=5.0 voxels
PASS - bridging removes the gap (CC agreement flags it) 


## Aggregation - macro-per-bone and subject-level

In [7]:
df = pd.DataFrame([
    {"subject": "s1", "side": "L", **{f"dice_{b}": v for b, v in zip(BONES, [0.9, 0.8, 0.7, 0.6])}},
    {"subject": "s1", "side": "R", **{f"dice_{b}": v for b, v in zip(BONES, [0.8, 0.8, 0.8, 0.8])}},
    {"subject": "s2", "side": "L", **{f"dice_{b}": v for b, v in zip(BONES, [1.0, 1.0, 1.0, 1.0])}},
])
df["dice_macro"] = df[[f"dice_{b}" for b in BONES]].mean(axis=1)
check("macro-per-bone == mean over bones", abs(df.loc[0, "dice_macro"] - 0.75) < 1e-9, f"macro={df.loc[0,'dice_macro']}")
subj = df.groupby("subject")["dice_macro"].mean()
check("subject-level == mean over that subject's knees", abs(subj["s1"] - (0.75 + 0.80) / 2) < 1e-9, f"s1={subj['s1']}")

PASS - macro-per-bone == mean over bones | macro=0.7500000000000001
PASS - subject-level == mean over that subject's knees | s1=0.7750000000000001


In [8]:
n_fail = sum(1 for _, ok, _ in RESULTS if not ok)
print("\n=== %d/%d checks passed ===" % (len(RESULTS) - n_fail, len(RESULTS)))
assert n_fail == 0, "%d metric-foundation checks FAILED" % n_fail


=== 23/23 checks passed ===
